# Analytical PMU-Based gSCR Identification

This tutorial implements the accumulated, non-iterative method in [Han et al. (2025)](https://doi.org/10.1049/gtd2.70026).

## Goal

Use a known three-port complex-symmetric network to verify the full PMU-increment → analytical admittance → gSCR workflow.

## Setup

The example is deterministic and does not require ANDES or PSASP. The larger cases use the same identifier.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gscr_demo import PMURun, generalized_scr, identify_symmetric_admittance

## Steps

### 1. Define a symmetric network and port capacities

In [2]:
conductance = np.array([
    [0.30, -0.05, -0.02],
    [-0.05, 0.25, -0.04],
    [-0.02, -0.04, 0.20],
])
susceptance = np.array([
    [8.0, -2.0, -1.0],
    [-2.0, 7.0, -1.5],
    [-1.0, -1.5, 6.0],
])
true_y = conductance - 1j * susceptance
capacities_pu = np.array([1.5, 1.0, 0.8])
pd.DataFrame(true_y, index=['P1', 'P2', 'P3'], columns=['P1', 'P2', 'P3'])

,P1,P2,P3
P1,0.30-8.00j,-0.05+2.00j,-0.02+1.00j
P2,-0.05+2.00j,0.25-7.00j,-0.04+1.50j
P3,-0.02+1.00j,-0.04+1.50j,0.20-6.00j


### 2. Generate PMU phasors

Current is calculated with the same retained-port reference direction used by the identification model.

In [3]:
rng = np.random.default_rng(20250701)
voltage = rng.standard_normal((160, 3)) + 1j * rng.standard_normal((160, 3))
current = voltage @ true_y.T
voltage.shape, current.shape

((160, 3), (160, 3))

### 3. Accumulate the analytical equations and solve once

In [4]:
identified = identify_symmetric_admittance([PMURun(voltage, current)])
identified.y_hat

array([[ 0.3 -8.j , -0.05+2.j , -0.02+1.j ],
       [-0.05+2.j ,  0.25-7.j , -0.04+1.5j],
       [-0.02+1.j , -0.04+1.5j,  0.2 -6.j ]])

### 4. Calculate and compare gSCR

In [5]:
direct_gscr = generalized_scr(true_y, capacities_pu).value
identified_gscr = generalized_scr(identified.y_hat, capacities_pu).value
relative_y_error = np.linalg.norm(identified.y_hat - true_y) / np.linalg.norm(true_y)

pd.DataFrame([{
    'direct_gscr': direct_gscr,
    'identified_gscr': identified_gscr,
    'absolute_gscr_error': abs(identified_gscr - direct_gscr),
    'relative_y_error': relative_y_error,
    'voltage_rank': identified.voltage_rank,
    'real_parameter_rank': identified.parameter_rank_real,
}])

,direct_gscr,identified_gscr,absolute_gscr_error,relative_y_error,voltage_rank,real_parameter_rank
0,3.593675,3.593675,8.881784e-16,2.340421e-16,3,12


## Checks

In [6]:
assert identified.voltage_rank == 3
assert identified.parameter_rank_real == 12
assert relative_y_error < 1e-12
assert abs(identified_gscr - direct_gscr) < 1e-12
print('All analytical identification checks passed.')

All analytical identification checks passed.


## Next Steps

Run `scripts/run_cepri36_psasp.py` for the PSASP-record workflow, `scripts/run_cepri36_andes.py` for the open ANDES reconstruction, and `scripts/run_il200.py` for the mixed synchronous-machine/IBR extension. See the case READMEs for port definitions and reference values.